In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.4 MB/s eta 0:00:00


In [ ]:
doc = Document("5_work_detail.docx")   //換成需要詢問的檔案

lines = []

for p in doc.paragraphs:
    text = p.text.strip()

    if text:
        lines.append(text)

In [ ]:
chunks = []

current_chapter = None
current_article = None

buffer = []

def save_chunk():
    global buffer

    if not buffer:
        return

    chunks.append({
        "chapter": current_chapter,
        "article": current_article,
        "content": "\n".join(buffer).strip()
    })

    buffer = []


for line in lines:

    line = line.strip()

    # 第一章、第二章...
    if re.match(r"^第[一二三四五六七八九十]+章", line):

        save_chunk()

        current_chapter = line
        current_article = None

        continue

    # 第一條、第二條...
    if re.match(r"^第[一二三四五六七八九十]+條", line):

        save_chunk()

        current_article = line
        buffer = [line]

        continue

    buffer.append(line)

save_chunk()

print("第一階段 chunks:", len(chunks))

第一階段 chunks: 65


In [ ]:
chunks[11]

{'chapter': '第二章 請假規則與流程',
 'article': '第一條 員工請假種類及准假日數以及工資給付規定如下：',
 'content': '第一條 員工請假種類及准假日數以及工資給付規定如下：\n公假：公假期間適逢例假日及休假日，本公司不另行補假。\n依政府法令規定參教育、公務應給公假等，依實際需要天數給予公假。\n公假期間工資照給。\n須檢附相關證明文件。\n公傷病假：\n員工因職業災害而致殘廢、傷害或疾病者，其治療、休養期間，給予公傷病假。\n假期間依勞動基準法第五十九條規定，給予原領工資補償。\n須檢附醫師診斷書。\n普通傷病假：\n員工因普通傷害、疾病或生理原因必須治療或休養者，得依下列規定請普通傷病假：\n未住院者，一年內合計不得超過三十日。\n住院者，二年內合計不得超過一年。\n未住院傷病假與住院傷病假二年內合計不得超過一年。\n普通傷病假超過規定之期限，經以事假或特別休假抵充後仍未痊癒者，經總經理核准者，則可辦理留職停薪，期間以一年為限。\n普通傷病假一年內合計未超過三十日部分，工資折半發給，其領有勞工保險普通傷病給付未達工資半數時，不足部分由本公司補足。\n請假三天及三天以上者，須檢附醫師診斷書。\n事假：\n員工因有事故必須親自處理者，得請事假，一年內合計不得超過十四天。\n事假期間不發給工資。事假超過規定日數得以未休之特別休假抵充。\n以小時為最小計假單位。\n婚假：\n員工本人結婚者，得請婚假八日(不含例假日)。\n婚假得分次申請，應自結婚之日前十日起三個月內請畢。\n婚假期間工資照給(不含例假日)。\n須附喜帖或有關文件申請。\n須一星期前提出申請。\n喪假：\n父母、養父母、繼父母、配偶喪亡者，給予喪假八日。\n祖父母、外祖父母、子女、配偶之祖父母、配偶之父母、配偶之養父母或繼父母喪亡者，給予喪假六日。\n兄弟姐妹喪亡者，給予喪假三日。\n喪假得依習俗於百日內分次申請。\n喪假期間工資照給(不含例假日)。\n須檢附死亡證明或相關證明文件。\n產假：\n女性員工分娩前後，應停止工作，給予產假八星期(含例假日)；妊娠三個月以上流產者，應停止工作，給予產假四星期(含例假日)。受僱工作在六個月以上者，停止工作期間工資照給；未滿六個月者減半發給。\n女性員工妊娠二個月以上未滿三個月流產者，應停止工作，給予產假一星期(含例假日)

第二階段：請假規則細切

例如：

公假：
公傷病假：
普通傷病假：
事假：
婚假：
喪假：
產假：
陪產假：
生理假：
特別休假：

In [ ]:
leave_keywords = [
    "公假",
    "公傷病假",
    "普通傷病假",
    "事假",
    "婚假",
    "喪假",
    "產假",
    "陪產假",
    "生理假",
    "特別休假"
]

final_chunks = []

for chunk in chunks:

    '''chapter = chunk.get("chapter", "")
    article = chunk.get("article", "")'''
    chapter = chunk.get("chapter") or ""
    article = chunk.get("article") or ""

    # 只處理
    # 第二章 請假規則與流程
    if (
        "請假規則與流程" not in chapter
        or "第一條" not in article
    ):
        final_chunks.append({
            "chapter": chapter,
            "article": article,
            "item": None,
            "content": chunk["content"]
        })
        continue

    lines2 = chunk["content"].split("\n")

    current_leave = None
    leave_buffer = []

    for line in lines2:

        line = line.strip()

        if not line:
            continue

        matched = False

        for leave_name in leave_keywords:

            if line.startswith(leave_name):

                matched = True

                if current_leave and leave_buffer:

                    final_chunks.append({
                        "chapter": chapter,
                        "article": article,
                        "item": current_leave,
                        "content": "\n".join(leave_buffer)
                    })

                current_leave = leave_name
                leave_buffer = [line]

                break

        if not matched:

            if current_leave:
                leave_buffer.append(line)

    # 最後一塊
    if current_leave and leave_buffer:

        final_chunks.append({
            "chapter": chapter,
            "article": article,
            "item": current_leave,
            "content": "\n".join(leave_buffer)
        })

建立 Metadata

In [ ]:
rag_chunks = []

for idx, chunk in enumerate(final_chunks):

    rag_chunks.append({
        "id": idx,

        "metadata": {
            "chapter": chunk["chapter"],
            "article": chunk["article"],
            "item": chunk["item"]
        },

        "content": chunk["content"]
    })

儲存 JSON

In [ ]:
with open(
    "employee_manual_chunks.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rag_chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print("完成")

完成


以上chunk切好了

1. 安裝套件

In [ ]:
!pip install -q sentence-transformers qdrant-client transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 27.3 MB/s eta 0:00:00


2. 載入 BGE-M3

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
embedding = embedding_model.encode(
    "特別休假有幾天？",
    normalize_embeddings=True
)

print(len(embedding))

1024


3. 建立 Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(":memory:")

建立 Collection：

In [ ]:
client.create_collection(
    collection_name="employee_manual",

    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE
    )
)

True

4. 寫入向量

In [ ]:
from qdrant_client.models import PointStruct

points = []

for chunk in rag_chunks:

    vector = embedding_model.encode(
        chunk["content"],
        normalize_embeddings=True
    ).tolist()

    points.append(
        PointStruct(
            id=chunk["id"],
            vector=vector,

            payload={
                "content": chunk["content"],
                **chunk["metadata"]
            }
        )
    )

上傳：

In [ ]:
client.upsert(
    collection_name="employee_manual",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

確認：

In [ ]:
client.count("employee_manual")

CountResult(count=82)

5. 搜尋函式

In [ ]:
def search_docs(
    query,
    top_k=5
):

    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    hits = client.query_points(
        collection_name="employee_manual",
        query=query_vector,
        limit=top_k
    )

    return hits.points

測試：

In [ ]:
results = search_docs(
    "特休有幾天"
)

for r in results:
    print("="*50)
    print(r.score)
    print(r.payload["content"][:300])

0.6966978106871935
特別休假(以週年制為計算基準):
工作半年以上未滿一年者給予三日。
工作一年以上未滿兩年者給予七日。
工作兩年以上未滿三年者給予十日。
工作三年以上未滿五年者給予十四日。
工作五年以上未滿十年者給予十五日。
工作十年以上，每一年加給一日，加至三十日為止。
新進人員服務滿半年以上者可申請預休(年資以報到日起算)，已預休但未滿應服務年資者，於離職時由薪資扣除已休日數。
特休假未休畢者，折現發給工資。
留職停薪者，當年度自留職停薪日起，不支給特休假。
以小時為最小計假單位。
0.5929686058170838
第三條 轉休必需加班工作滿 4 小時方得申請轉休半天，逾 8 小時者，一律以轉
休一天之方式辦理，且此轉休必需於自發生日之隔日起 90 天內有效，逾時不休者，視為棄權論。
0.5811316228127237
事假期間不發給工資。事假超過規定日數得以未休之特別休假抵充。
以小時為最小計假單位。
0.577693111337563
事假：
員工因有事故必須親自處理者，得請事假，一年內合計不得超過十四天。
0.5719665745713447
公假：公假期間適逢例假日及休假日，本公司不另行補假。
依政府法令規定參教育、公務應給公假等，依實際需要天數給予公假。


6. 載入 Qwen

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

7. 建立 RAG Prompt

In [ ]:
def build_context(results):

    texts = []

    for r in results:

        texts.append(
            f"""
章節:
{r.payload.get('chapter')}

條文:
{r.payload.get('article')}

項目:
{r.payload.get('item')}

內容:
{r.payload.get('content')}
"""
        )

    return "\n\n".join(texts)

8. RAG 問答函式

In [ ]:
def ask_rag(question):

    docs = search_docs(
        question,
        top_k=5
    )

    context = build_context(docs)

    prompt = f"""
你是公司員工手冊查詢助手。

請只根據提供的文件內容回答。

若文件沒有答案，
請回答：

「員工手冊中未找到相關資訊」

----------------

文件內容：

{context}

----------------

問題：

{question}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1
    )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer

9. 測試

In [ ]:
print(
    ask_rag(
        "特休有幾天？"
    )
)

根據提供的文件內容，員工的特別休假（特休）的具體天數是按照工作年限來計算的，具體如下：

- 工作半年以上未滿一年者給予三日。
- 工作一年以上未滿兩年者給予七日。
- 工作兩年以上未滿三年者給予十日。
- 工作三年以上未滿五年者給予十四日。
- 工作五年以上未滿十年者給予十五日。
- 工作十年以上，每一年加給一日，加至三十日為止。

因此，特別休假的天數並不是固定的，而是根據員工的工作年限來決定的。


In [ ]:
print(
    ask_rag(
        "婚假有幾天？"
    )
)

婚假為八天。


In [ ]:
print(
    ask_rag(
        "試用期多久？"
    )
)

試用期為三個月。
